In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

# 1) Read the dataset
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)




In [ ]:
# Task 2: Write your code here:

 #2- Inspect first few rows
print("First 5 rows:")
print(df.head())
print("-" * 50)

In [ ]:
# Task 3: Write your code here:
# 3- Dataset information
print("Dataset info:")
df.info()
print("-" * 50)

In [ ]:
# Task 4: Write your code here:
# 4- Statistical description
print("Statistical description:")
print(df.describe())

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import StandardScaler
import pandas as pd


target_col = "target"


missing_cols = df.columns[df.isnull().any()]

if len(missing_cols) > 0:
    print("Handling missing values in:", list(missing_cols))
    for col in missing_cols:
        if df[col].dtype == "object":
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())
else:
    print("No missing values found")





In [ ]:
# Task 2: Write your code here:


dup_count = df.duplicated().sum()
if dup_count > 0:
    df = df.drop_duplicates()
    print(f"Removed {dup_count} duplicate rows")
else:
    print(" No duplicate rows found")


In [ ]:
# Task 3: Write your code here:


cat_cols = df.select_dtypes(include=["object"]).columns

if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    print("Categorical variables encoded using One-Hot Encoding")
else:
    print("No categorical variables to encode")


In [ ]:
# Task 4: Write your code here:

X = df.drop(columns=[target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("Feature scaling applied to numerical features")
print("Final feature shape:", X.shape)


In [ ]:
# Task 5: Write your code here:

target_counts = y.value_counts(normalize=True)
print("Target distribution:")
print(target_counts)

if target_counts.min() < 0.4:
    print("Target is imbalanced")
else:
    print(" Target is balanced")

In [ ]:
from catboost import   # idont know how to import catboost and i do my bist to solve the modeling with it
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np


# 1- Split features and target
X = X.values
y = y.values

print("Features and target prepared")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("-" * 40)


In [ ]:
# Task 2,3,4,5: Write your code here:


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []


for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

    print(f"Fold {fold} F1-score:", round(f1, 4))




avg_f1 = np.mean(f1_scores)
print("Average F1-score across all folds:", round(avg_f1, 4))

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import numpy as np


final_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    verbose=0,
    random_state=42
)

final_model.fit(X, y)


importances = final_model.get_feature_importance()
feature_names = df.drop(columns=["target"]).columns

# Sort by importance
indices = np.argsort(importances)

plt.figure(figsize=(8, 6))
plt.barh(feature_names[indices], importances[indices])
plt.xlabel("Feature Importance")
plt.title("CatBoost Feature Importance")
plt.show()




In [ ]:
# Task 2: Write your code here:


golden_idx = np.argmax(importances)
golden_feature = feature_names[golden_idx]

print(" Golden Feature (Most Important):", golden_feature)
print("Importance Score:", round(importances[golden_idx], 4))

In [ ]:
# Task Bonus: Write your code here: